# Notebook 4b: Population + Unemployment

Population data: single year of age (0, 1, 2... 90)
Unemployment data: LFS age bands (16-17, 18-24, 25-34, 35-49, 50-64,65+)

Step 1: Add LFS age band labels to population data
Step 2: SUM population whithin each band per year and sex
Step 3: JOIN on Year + Age_Group
Step 4: Derive Est_Unemployed_Count = Population x Rate / 100

In [2]:
import pandas as pd
import numpy as np
import sqlite3

CLEAN_DIR = '../../data/clean/'
DE_PATH = '../../data/uk_population.db'

IndentationError: expected an indented block after 'else' statement on line 16 (2678816732.py, line 17)

In [9]:
df_pop = pd.read_csv(f'{CLEAN_DIR}table3_uk_population_long.csv')

def assign_lfs_band(age):
    if age <= 15: return '0-15 Below working age'
    elif age <= 17: return '16-17'
    elif age <= 24: return '18-24'
    elif age <= 34: return '25-34'
    elif age <= 49: return '35-49'
    elif age <= 64: return '50-64'
    else: return '65+'

df_pop['LFS_Age_Group'] = df_pop['Age'].apply(assign_lfs_band)
df_pop.to_csv(f'{CLEAN_DIR}table3_uk_population_long.csv', index=False)
print("Done. Unique bands:", sorted(df_pop['LFS_Age_Group'].unique()))

Done. Unique bands: ['0-15 Below working age', '16-17', '18-24', '25-34', '35-49', '50-64', '65+']


In [19]:
LFS_BANDS = ['16-17', '18-24', '25-34', '35-49', '50-64', '65+']

pop_lfs = (
    df_pop[df_pop['LFS_Age_Group'].isin(LFS_BANDS)]
    .groupby(['Year', 'Sex','LFS_Age_Group'])['Population']
    .sum()
    .reset_index()
    .rename(columns={'LFS_Age_Group':'Age_Group'})
)

print(pop_lfs[(pop_lfs['Year']==2023)&(pop_lfs['Sex']=='Persons')].to_string(index=False))

 Year     Sex Age_Group  Population
 2023 Persons     16-17     1595102
 2023 Persons     18-24     5712374
 2023 Persons     25-34     9219841
 2023 Persons     35-49    13190249
 2023 Persons     50-64    13340361
 2023 Persons       65+    12317068


In [23]:
df_unemp = pd.read_csv(f'{CLEAN_DIR}unemployement_by_age_annual.csv')

pop_persons = pop_lfs[pop_lfs['Sex']=='Persons'][['Year','Age_Group','Population']]
pop_m = pop_lfs[pop_lfs['Sex']=='Males']
[['Year','Age_Group','Population']].rename(columns={'Population':'Pop_Male'})
pop_f = poplfs[pop_lfs['Sex']=='Females']
[['Year','Age_Group','Population']].rename(columns={'Population':'Pop_Female'})

df_combined = (df_unemp
               .merge(pop_persons, on=['Year','Age_Group'], how='inner')
               .merge(pop_m, on=['Year','Age_Group'], how='left')
               .merge(pop_f, on=['Year','Age_Group'], how='left')
              )

df_combined['Est_Unemployed_Count'] = (df_combined['Population'] * df_combined['Est_Unemployed_Rate_All'] /100).round(0).astype(int)
df_combined['Est_Unemployed_Male'] = (df_combined['Pop_Male'] * df_combined['Unemployed_Rate_Male'] /100).round(0).astype(int)
df_combined['Est_Unemployed_Female'] = (df_combined['Pop_Female'] * df_combined['Unemployed_Rate_Female'] /100).round(0).astype(int) 
df_combined['Est_Employed_Count'] = df_combined['Population'] - df_combined['Est_Unemployed_Count']

df_combined = df_combined.sort_values(['Year','Age_Group']).reset_index(drop=True)
df_combined.to_csv(f'{CLEAN_DIR}combined_population_unemployement.csv', index=False)

print(f"Saved.Shape:{df_combined.shape}")
print(df_combined[df_combined['Year']==2023][['Age_Group','Unemployement_Rate_All','Population','Est_Unemployed_Count']].to_string(index=False))

FileNotFoundError: [Errno 2] No such file or directory: '../../data/clean/unemployement_by_age_annual.csv'

In [27]:
import pandas as pd

LEAN_DIR = '../data/clean/'

df_pop = pd.read_csv(f'{CLEAN_DIR}table3_uk_population_long.csv')

pop_16_24 = (
    df_pop[
        df_pop['Age'].between(16, 24) &
        df_pop['Sex'].isin(['Males','Females']) &
        df_pop['Year'].between(1990,2023)
        ]
    .groupby(['Year', 'Sex'])['Population']
    .sum()
        .reset_index()
        )
pop_16_24['Age_Group'] = '16-24'

print(f"Shape: {pop_16_24.shape}")
print("\nSample - key years:")
for yr in [1990, 2009, 2020, 2023]:
    print(f" {yr}:", pop_16_24[pop_16_24['Year']==yr]
          [['Sex','Population']].values.tolist())

Shape: (68, 4)

Sample - key years:
 1990: [['Females', 3802284], ['Males', 3913970]]
 2009: [['Females', 3669723], ['Males', 3715199]]
 2020: [['Females', 3493695], ['Males', 3574327]]
 2023: [['Females', 3569215], ['Males', 3738261]]


In [34]:
lfs_rates = {
    1990: {'Males':14.2, 'Females':11.8},
    1991: {'Males':17.1, 'Females':13.9},
    1992: {'Males':19.8, 'Females':15.9},
    1993: {'Males':21.0, 'Females':16.8},
    1994: {'Males':19.7, 'Females':15.15},
    1995: {'Males':18.8, 'Females':14.9},
    1996: {'Males':17.5, 'Females':13.9},
    1997: {'Males':15.8, 'Females':12.5}, 
    1998: {'Males':14.2, 'Females':11.6},
    1999: {'Males':13.1, 'Females':10.8},
    2000: {'Males':12.1, 'Females':10.2},
    2001: {'Males':12.2, 'Females':10.3},
    2002: {'Males':12.6, 'Females':10.8},
    2003: {'Males':13.1, 'Females':11.2},
    2004: {'Males':13.3, 'Females':11.5},
    2005: {'Males':13.8, 'Females':11.8},
    2006: {'Males':14.2, 'Females':12.1},
    2007: {'Males':15.1, 'Females':13.0},
    2008: {'Males':17.8, 'Females':14.9},
    2009: {'Males':22.5, 'Females':18.4},
    2010: {'Males':22.8, 'Females':18.8},
    2011: {'Males':23.0, 'Females':19.1},
    2012: {'Males':22.5, 'Females':18.5},
    2013: {'Males':21.2, 'Females':17.4},
    2014: {'Males':18.0, 'Females':14.8},
    2015: {'Males':15.5, 'Females':12.8},
    2016: {'Males':14.5, 'Females':12.0},
    2017: {'Males':13.5, 'Females':11.2},
    2018: {'Males':13.0, 'Females':10.9},
    2019: {'Males':12.8, 'Females':10.7},
    2020: {'Males':16.2, 'Females':13.5},
    2021: {'Males':15.5, 'Females':13.0},
    2022: {'Males':12.8, 'Females':10.6},
    2023: {'Males':13.2, 'Females':10.9},
    }

pop_16_24['Unemployment_Rate_Pct'] = pop_16_24.apply(
    lambda r:lfs_rates[r['Year']][r['Sex']], axis=1
)
pop_16_24['Unemployed_Count'] = (pop_16_24['Population'] * pop_16_24['Unemployment_Rate_Pct']/100).round(0).astype(int)

pop_16_24['Employed_Count'] = pop_16_24['Population'] - pop_16_24['Unemployed_Count']

pop_16_24['Source'] = 'ONS Mid-Year Estimates + ONS LFS A05 SA'

df_youth = pop_16_24[['Year','Age_Group','Sex','Population',
'Unemployment_Rate_Pct','Unemployed_Count',
'Employed_Count','Source']].sort_values(['Year','Sex']).reset_index(drop=True)

print(f"Dataset shape: {df_youth.shape}")
print(f"\nFull view - 2023:")
print(df_youth[df_youth['Year']==2023].to_string(index=False))
print(f"\nFull view - 2009 (recession peak):")
print(df_youth[df_youth['Year']==2009].to_string(index=False))

Dataset shape: (68, 8)

Full view - 2023:
 Year Age_Group     Sex  Population  Unemployment_Rate_Pct  Unemployed_Count  Employed_Count                                  Source
 2023     16-24 Females     3569215                   10.9            389044         3180171 ONS Mid-Year Estimates + ONS LFS A05 SA
 2023     16-24   Males     3738261                   13.2            493450         3244811 ONS Mid-Year Estimates + ONS LFS A05 SA

Full view - 2009 (recession peak):
 Year Age_Group     Sex  Population  Unemployment_Rate_Pct  Unemployed_Count  Employed_Count                                  Source
 2009     16-24 Females     3669723                   18.4            675229         2994494 ONS Mid-Year Estimates + ONS LFS A05 SA
 2009     16-24   Males     3715199                   22.5            835920         2879279 ONS Mid-Year Estimates + ONS LFS A05 SA


In [36]:
import sqlite3

df_youth.to_csv(f'{CLEAN_DIR}youth_unemployment_16_24_real.csv', index=False)
print("Saved: youth_unemployment_16_24_real.csv")

conn = sqlite3.connect('../../data/uk_population.db')
df_youth.to_sql('youth_unemployment', conn, if_exists='replace', index=False)

tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(f"\nDatabase tables: {tables['name'].tolist()}")

validation = pd.read_sql("""
SELECT Year,
SUM(Population) AS Total_Pop_16_24,
SUM(Unemployed_Count) AS Total_Unemployed,
ROUND(SUM(Unemployed_Count)*100.0/SUM(Population),1) AS Over_Rate
FROM youth_unemployment
GROUP BY Year ORDER BY Year
""", conn)
print(f"\nHeadline trend:")
print(validation.to_string(index=False))

Saved: youth_unemployment_16_24_real.csv

Database tables: ['population', 'total_population', 'youth_unemployment']

Headline trend:
 Year  Total_Pop_16_24  Total_Unemployed  Over_Rate
 1990          7716254           1004454       13.0
 1991          7491104           1162859       15.5
 1992          7234725           1293210       17.9
 1993          6980056           1321007       18.9
 1994          6749114           1177640       17.4
 1995          6613551           1115616       16.9
 1996          6495229           1020817       15.7
 1997          6408500            907663       14.2
 1998          6357614            820735       12.9
 1999          6363056            760922       12.0
 2000          6383609            712273       11.2
 2001          6503380            732237       11.3
 2002          6647035            778370       11.7
 2003          6790463            825705       12.2
 2004          6938045            860979       12.4
 2005          7058561            9